# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

# 실험 계획

자료가 목차와 카테고리를 기반으로 탐색하면 유리한 구조로 되어 있으므로,

메타데이터를 활용하여 카테고리에 대한 정보를 가진 데이터와 그렇지 않은 데이터로

RAG의 성능을 비교할 계획이다.

In [54]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


In [55]:
import re
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

import pdfplumber
from img2table.document import PDF
from img2table.ocr import TesseractOCR

In [56]:
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

for text in docs:
    text.metadata = {"page": text.metadata["page"]}

### 표 데이터

In [57]:
pdf = PDF(
    PDF_PATH, 
    detect_rotation=False,
    pdf_text_extraction=True
)

ocr = TesseractOCR(n_threads=1, lang="eng")

TABLES_BY_PAGE = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=40
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [58]:
# 표 데이터 metadata에 merge
def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df


to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        docs[page_num].metadata["table"] = {
            f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
        }

        for table in tables:
            table = trim_table(table.df)

    else:
        docs[page_num].metadata["table"] = None
        to_delete_list.append(page_num)


for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

### 텍스트 데이터

In [59]:
# 문서 구조 기반 페이지 정리
DOCS_ABSTRACT = docs[:10]
DOCS_INDEX = docs[10:15]
DOCS_REVISED = docs[15:37]
DOCS_CONCRETE = docs[37:]

part_list = [DOCS_ABSTRACT, DOCS_INDEX, DOCS_REVISED, DOCS_CONCRETE]
page_range_list = [range(10), range(10, 15), range(15, 37), range(37, 426)]

### 요약 페이지(abstract)

In [60]:
# 메타데이터(chapter_title)
with pdfplumber.open(PDF_PATH) as pdf:
    for page in DOCS_ABSTRACT:
        page_num = page.metadata["page"] 

        head = pdf.pages[page_num].within_bbox((0, 0, 538, 130))
        tail = pdf.pages[page_num].within_bbox((0, 131, 538, 737))

        if head.extract_text():
            target_text = head.extract_text().replace("\n", " ")

        page.metadata["chapter_title"] = target_text
        page.page_content = tail.extract_text()

In [61]:
print(DOCS_ABSTRACT[0])

page_content='2 4
원천징수의무자를
위
2 0 한
일 하나는 제대로 하는,
국민께 인정받는 국세청
연말정산
신고안내
2024. 12.
간소화 서비스 맞춤형 안내 일괄제공 서비스' metadata={'page': 0, 'table': None, 'chapter_title': '발 간 등 록 번 호 11-1210000-000072-10'}


### 목차 페이지(index)

In [62]:
# split 구분: 대제목
for page in DOCS_INDEX[:-1]:
    split = page.page_content.split("\n")
    index_list = [sentence for sentence in split if "·" in sentence]
    page.page_content = "\n".join(index_list)


chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separators=[r"\n(?=[가-힣])"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_INDEX = chapter_splitter.split_documents(DOCS_INDEX)

# 메타데이터(chapter_title)
for chapter in DOCS_INDEX:
    chapter.metadata["chapter_title"] = re.sub(
        r"[\s·.]{2,}\d+", "", 
        chapter.page_content.strip().split("\n")[0]
    )

In [63]:
print(DOCS_INDEX[0])

page_content='2024년 귀속 연말정산 개정세법 요약 ·······························1' metadata={'page': 10, 'table': None, 'chapter_title': '2024년 귀속 연말정산 개정세법 요약'}


### 개정안 페이지(revised)

In [64]:
# split 구분: 개정안 항목
chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=[r"\d+\s+.*?\n\(.*"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_REVISED = chapter_splitter.split_documents(DOCS_REVISED)

# 메타데이터(chapter_title)
del_list = list()
for chapter in DOCS_REVISED:
    if chapter.page_content == "원천징수의무자를 위한 \n2024년 연말정산 신고안내":
        del_list.append(chapter)
    elif chapter.page_content == "01. 2024년 귀속 연말정산 개정세법 요약":
        del_list.append(chapter)

    else:
        match = re.search(r"^\d+\s+(.*?)\n\(.*", chapter.page_content, re.DOTALL)
        if match:
            chapter_title = match.group(1).strip()
            chapter.metadata["chapter_title"] = chapter_title
            chapter.page_content = chapter.page_content.replace(chapter_title, "")

for del_ in del_list:
    DOCS_REVISED.remove(del_)

In [65]:
print(DOCS_REVISED[1])

page_content='1  
(소득세법 제12조 제3호 마목, 같은 법 시행령 제10조의2)
<개정취지> 육아휴직 지원
종          전 개          정
▢ 근로소득에서 비과세되는 육아휴직 급여·수당 ▢ 비과세 소득 확대
○ ｢고용보험법｣에 따라 받는 육아휴직급여 ○ (좌  동)
○ 공무원 또는 ｢사립학교교직원 연금법｣, ｢별정우체국법｣을
적용받는 사람이 관련 법령에 따라 받는 육아휴직수당
<추  가>    - 사립학교 직원이 사립학교 정관 등에 의해 지급받는 
월 150만원 이하의 육아휴직수당
<적용시기> 2024.1.1. 이후 지급받는 분부터 적용
' metadata={'page': 16, 'table': None, 'chapter_title': '육아휴직수당 비과세 적용대상 확대 및 범위 규정'}


### 상세 페이지(concrete)

In [66]:
concrete_map = {
    "2024년 귀속 연말정산 중점 추진사항": range(37, 62),
    "근로소득 연말정산": range(62, 224),
    "연말정산 종합사례 및 서식 작성방법": range(224, 348),
    "사업소득·연금소득 연말정산": range(348, 368),
    "종교인 소득 연말정산": range(368, 382),
    "연말정산 관련 서비스": range(382, 400),
    "연말정산 간소화 서비스": range(400, 409),
    "간소화자료 일괄제공 서비스": range(409, 414),
    "맞벌이부부 연말정산": range(414, 415),
    "연말정산 주요 용어 설명": range(415, 419),
    "소득·세액공제신고서 첨부서류": range(419, 426),
}

tmp_num_map = dict()
for key, value in concrete_map.items():
    for num in value:
        tmp_num_map[num] = key

# 메타데이터(chapter_title)
for page in DOCS_CONCRETE:
    page.metadata["chapter_title"] = tmp_num_map[page.metadata["page"]]

pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

In [67]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

'https://huggingface.co/microsoft/tapex-base-finetuned-wikisql'

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.